In [1]:
# %%

#------------------------------------------------ Begin_Librairie ----------------------------------------



import datetime

import pandas as pd

from pandas import ExcelWriter

from selenium import webdriver


from time import sleep

import os
import requests
from bs4 import BeautifulSoup



In [2]:

# %%

#------------------------------------------------ Begin_ fileName ----------------------------------------
regulatorName = 'BO APS'
print(f"Running {regulatorName} Web Scraping Tool v.1.1")

now=datetime.datetime.now()

filename= f'{regulatorName} SQL Ready {str(now).replace(":",".")[:-7]}.xlsx'

scriptfolder = f"C:\\Users\\wuj1\\OneDrive - Moody's\\Desktop\\Regulator\\{regulatorName}"
#scriptfolder=os.path.dirname(os.path.abspath(__file__)) ## to decomment for the production environment

os.chdir(scriptfolder)

tempfolder=os.path.join(scriptfolder, 'tempfolder') #if files are downloaded during the process



if os.path.exists(tempfolder):

    for rem in os.listdir(tempfolder):

        os.remove(os.path.join(tempfolder, rem))

else:

    os.mkdir(tempfolder)





Running BO APS Web Scraping Tool v.1.1


In [3]:

# %%

#------------------------------------------------ Begin_Fouction ----------------------------------------


def bourange_same_length_array(sqldict) :

    maxlen = len(sqldict['ListProcessDate'])

    for key, val in sqldict.items():

        if len(sqldict[key]) != maxlen:

            empty = []

            total_empty = maxlen - len(sqldict[key])

            for i in range(total_empty):

                empty.append('')

            sqldict[key]=sqldict[key]+empty

    return sqldict




In [4]:
# %%

#------------------------------------------------ Begin_Variable ----------------------------------------

regdict = {
    regulatorName+' 1' : 'https://www.aps.gob.bo/index.php/pensiones/entidades-fiscalizadas',
    regulatorName+' 2' : 'https://www.aps.gob.bo/index.php/seguros/entidades-fiscalizadas',
    }

Typology = {

            regulatorName+" 1": "Entidades fiscalizadas Pensiones",
            regulatorName+" 2": "Entidades Fiscalizadas Seguros",

            }



sqldict={'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [], 'Name': [], 'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [], 

          'InternalID_2_type': [], 'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [], 'License_Type': [], 'Address_1': [], 'Address_2': [], 'City': [], 

          'Zip': [], 'Cntry': [], 'Phone': [], 'Fax': [], 'Website': [], 'Email': [], 'RegulationType': [], 'RegulationTypeCode': [], 'RegulationDate': [], 'CancellationDate': [], 

          'RegCtry': [], 'RegCode' : [], 'ListCode': [], 'ListLanguage': [], 'ListValidityDate': [], 'ListName': [], 'ListProcessDate': [], 'LEI Code': [], 'BIC SWIFT Code': [], 'Name - Mother Company': [],

          'Address_1 - Mother company': [], 'Address_2 -  Mother company': [], 'City - Mother company': [], 'Zip - Mother company': [], 'Cntry - Mother company': [], 

          'Phone - Mother company': [], 'Check': []}





processdate = now.strftime('%Y-%m-%d')



In [5]:

# %%

#------------------------------------------------ Begin_chromedriver ----------------------------------------

#Starting Chrome driver, set to download files in tempfolder

chromeOptions = webdriver.ChromeOptions()

prefs = {"plugins.always_open_pdf_externally": True,

		 "download.prompt_for_download": False,

		 "download.default_directory" : tempfolder}

chromeOptions.add_experimental_option("prefs",prefs)

#driver = webdriver.Chrome(service=ChromeService(ChromeDriverManager().install()), options=chromeOptions )

driver = webdriver.Chrome(options=chromeOptions)

driver.maximize_window()



In [6]:
# %%

#------------------------------------------------ Begin_Main ----------------------------------------

for k, reg in enumerate(regdict):

    print(f"[INFO] : Working {k+1}/{len(regdict)} _({reg})_ ")

    sleep(3)
    # Fetch the page content
    response = requests.get(regdict[reg])
    response.encoding = 'utf-8'  # Ensure correct encoding
    soup = BeautifulSoup(response.text, 'html.parser')

    # Find all <div class='sppb-addon-content'>
    target_divs = soup.find_all('div', class_='sppb-addon-content')

    # Extract <li> text and <a> hrefs within those divs
    extracted_data = []
    for div in target_divs:
        li_elements = div.find_all('li')
        for li in li_elements:
            li_text = li.get_text(strip=True)
            a_tag = li.find('a')
            href = a_tag['href'] if a_tag and 'href' in a_tag.attrs else ''
            extracted_data.append({'li_text': li_text, 'href': href})

    # Display example output
    for item in extracted_data[:]:  # Show first 10 for brevity
        name_ = item['li_text']
        website_ = item['href']
        sqldict['Name'].append(name_)
        sqldict['Website'].append(website_)
        sqldict['ListProcessDate'].append(processdate)
        sqldict['RegCtry'].append(reg.split()[0])
        sqldict['RegCode'].append(reg.split()[1])
        sqldict['ListCode'].append(reg.split()[2])
        sqldict['ListName'].append(Typology[reg])
        sqldict['RegulationType'].append('Regulated')
    sqldict=bourange_same_length_array(sqldict)


[INFO] : Working 1/2 _(BO APS 1)_ 
[INFO] : Working 2/2 _(BO APS 2)_ 


In [7]:
		
#------------------------------------------------ Begin_writer and save df to excel  ----------------------------------------

os.chdir(scriptfolder)
df=pd.DataFrame(sqldict)
df.to_excel(filename, index=False)


sleep(3)

driver.quit()